In [ ]:
import pandas as pd
import re
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def covers_selector(data, attribute, value):
    row = data[attribute].to_numpy()
    if pd.isnull(value):
        return pd.isnull(row)
    return row == value
def covers_subgroup(data, pattern):
    return np.all([covers_selector(data,sel[0],sel[1]) for sel in pattern] , axis=0)

### Load dataset and filter by cancer type (in this case lung)

In [ ]:
df = pd.read_csv(f"./lung_cancer.csv",index_col=0)
class_name = "Class"
# Class value mapping: Mortality -> 1, Incidence -> 2
df[class_name] = 1
df = df.astype(str)

### Define the subgroup/population to analyze

In [ ]:
# Define the subgroup we want to get information about and get the corresponding rows
pattern_description = "Start_Year==2020"
parts = re.split(r'\s+AND\s+', pattern_description)
pattern = r'\s*(.*?)\s*(==|>=|<=|>|<)\s*(.*)\s*'
result = []
for p in parts:
    left, op, right = re.match(pattern, p).groups()
    # rule: if it is == we do not show it, but keep the value as is
    if op == "==":
        result.append((left, right))
    else:
        result.append((left, f"{op} {right}"))

# Get the rows corresponding to the subgroup
ic = covers_subgroup(df,result)
data_filter_subgroup = df.loc[ic]

In [ ]:
data_filter_subgroup

### Variables to get information about

In [ ]:
vars_to_search_info = [
    "Nickel(Ni)",
    "Arsenic(As)",
    "Particulate_Matter(PM2.5)",
    "Cadmium(Cd)",
    "Lead(Pb)",
    "Sex",
    "annual_mean_temp"
]

### Query the variable or variables to see descriptives

In [ ]:
vc = data_filter_subgroup[vars_to_search_info].value_counts()
vc

In [ ]:
df_var = data_filter_subgroup[vars_to_search_info]

total = len(df_var)

print(total)

for col in df_var.columns:
    pct = df_var[col].value_counts(normalize=True).mul(100).round(2)
    print(pct)
    print("\n")

In [ ]:
""" # We can define whether to group values of the queried variable and get rows corresponding to the requierement
values = list(set(vc.index.tolist()) - set(['(0.28, 0.4]']))
df_filtrado = data_filter_subgroup[data_filter_subgroup[vars_to_search_info[0]].isin(values)] """

## TreeMap: importance of each variable

In [ ]:
import squarify
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

# Weight of each variable
variable_weight = {}

for var in vars_to_search_info:
    # Value counts for each variable
    variable_count = data_filter_subgroup[var].value_counts()
    
    # Greater deviation = the variable creates larger and heavier groups
    # compared with smaller ones
    if len(variable_count) > 1:
        variable_weight[var] = variable_count.std()
    else:
        # If there is only one category, calculate the maximum standard
        # deviation it would have if the remaining categories in the
        # original dataset had a value of 0
        k = df[var].nunique()
        if k > 1:
            variable_weight[var] = variable_count.sum() / (k ** 0.5)
        else:
            variable_weight[var] = 0.0


# Sort variables from highest to lowest weight
sorted_variables = sorted(
    variable_weight,
    key=variable_weight.get,
    reverse=True
)

main_variable = sorted_variables[0]


print(f"--> Ranking by weight: {sorted_variables}")
print(f"--> Dominant variable: {main_variable}\n")

# Total importance
total_importance = sum(variable_weight.values())

# Prepare data lists (size = importance / total importance)
variable_names = list(variable_weight.keys())
weights = list(variable_weight.values())

if total_importance == 0:
    sizes = [1.0 / len(weights) for _ in weights]
else:
    sizes = [weight / total_importance for weight in weights]

percentages = [size * 100 for size in sizes]

# Sort from highest to lowest "weight" for a better layout
sorted_data = sorted(
    zip(sizes, variable_names, percentages, weights),
    reverse=True
)

filtered_data = [d for d in sorted_data if d[0] > 0]

# If, for any reason, all filtered data are removed
# (all importance values are 0), force equal sizes
if not filtered_data:
    num_variables = len(sorted_data)
    filtered_data = [
        (1.0 / num_variables, d[1], 100.0 / num_variables, d[3])
        for d in sorted_data
    ]

# Unpack the cleaned and safe data
sizes_sorted, names_sorted, percentages_sorted, weights_sorted = zip(
    *filtered_data
)

# Colors (gradient)
norm = mcolors.Normalize(
    vmin=min(sizes_sorted),
    vmax=max(sizes_sorted)
)

base_cmap = plt.cm.Blues

cmap = mcolors.LinearSegmentedColormap.from_list(
    "truncated_Blues",
    base_cmap(np.linspace(0.25, 0.75, 256))
)

# Colors for the final array
colors = [cmap(norm(size)) for size in sizes_sorted]

# Labels
labels = []

for name, weight, percentage in zip(
    names_sorted,
    weights_sorted,
    percentages_sorted
):
    text = f"{name}\n{weight:.1f}\n({percentage:.1f}%)"
    labels.append(text)

# Treemap
fig, ax = plt.subplots(figsize=(12, 7))

squarify.plot(
    sizes=sizes_sorted,
    label=labels,
    color=colors,
    alpha=0.85,
    text_kwargs={
        "fontsize": 12,
        "fontweight": "bold",
        "color": "#1A1A1A",
        "wrap": True
    },
    edgecolor="white",
    linewidth=3,
    ax=ax
)

# Title
ax.set_title(
    "Relative importance of each variable",
    fontsize=15,
    fontweight="bold",
    pad=15
)

ax.axis("off")

# Visual legend
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

cbar = fig.colorbar(
    sm,
    ax=ax,
    orientation="horizontal",
    fraction=0.04,
    pad=0.02
)

cbar.set_label(
    "Proportion of importance (Importance of variable / Total)",
    fontsize=11,
    fontweight="bold"
)

plt.tight_layout()
plt.show()